In [3]:
# Scientific Image Forgery Detection
# MOD006567 - Applications of Machine Learning
# Anglia Ruskin University
#
# This notebook builds AgenticForensicNet — a three-agent segmentation model
# for detecting pixel-level forgeries in biomedical publication images.
# Dataset: Recod.AI / LUC Kaggle 

import os

# this needs to go before torch loads otherwise the GPU allocator
# is already initialised and completely ignores it
# saved us around 1-2 GB on the T4 which we really needed
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from PIL import Image
import glob

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device           : {device}")
print(f"GPU name         : {torch.cuda.get_device_name(0)}")
print(f"GPU total memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device           : cuda
GPU name         : Tesla T4
GPU total memory : 15.6 GB


In [4]:
# Block 1 - Dataset loading and stratified splitting
#
# One thing we were careful about here is HOW we split the data.
# A plain random split doesn't account for forgery size — some images
# only have 0.1% of pixels forged (nearly impossible to detect) while
# others have 20% forged (much easier). If the easy ones all landed in
# validation by chance, our IoU score would look better than it really is.
#
# Our fix: compute forgery percentage per mask, group into 4 density bins,
# then split each bin separately so every set gets a proportional mix
# of micro, small, medium and large forgeries.

data_dir = "/kaggle/input/competitions/recodai-luc-scientific-image-forgery-detection"

# grab all file paths — recursive=True walks into subdirectories
authentic_paths = glob.glob(f"{data_dir}/train_images/authentic/**/*.png", recursive=True)
forged_paths    = glob.glob(f"{data_dir}/train_images/forged/**/*.png",    recursive=True)
mask_paths      = glob.glob(f"{data_dir}/train_masks/**/*.npy",            recursive=True)

print("Dataset loaded:")
print(f"  Authentic images   : {len(authentic_paths)}")
print(f"  Forged images      : {len(forged_paths)}")
print(f"  Ground truth masks : {len(mask_paths)}")


# step 1 - work out what percentage of each image is actually forged
forgery_percentages = []
forged_image_paths  = []

for mask_path in mask_paths:

    # masks can come in different shapes depending on how they were saved
    # squeeze handles [H,W], [1,H,W] and [H,W,1] all at once
    mask = np.load(mask_path)
    mask = np.squeeze(mask)
    if mask.ndim == 3:
        mask = mask[0]

    pct = (mask > 0).sum() / mask.size * 100
    forgery_percentages.append(pct)

    # rebuild the image path from the mask path
    img_path = (mask_path
                .replace("train_masks", "train_images/forged")
                .replace(".npy", ".png"))
    forged_image_paths.append(img_path)

forgery_percentages = np.array(forgery_percentages)

print(f"\nForgery density stats:")
print(f"  Average : {forgery_percentages.mean():.2f}%")
print(f"  Min     : {forgery_percentages.min():.4f}%")
print(f"  Max     : {forgery_percentages.max():.2f}%")
# around 4.89% average — this confirms the severe 150:1 class imbalance


# step 2 - bin images by forgery size
# bin 1: 0-0.5%   tiny forgeries, hardest to detect
# bin 2: 0.5-1%   small patch
# bin 3: 1-5%     medium, noticeable if you look
# bin 4: 5-100%   large, easier to find
bins         = [0, 0.5, 1.0, 5.0, 100.0]
forgery_bins = np.digitize(forgery_percentages, bins)

for b in range(1, 6):
    count = (forgery_bins == b).sum()
    print(f"  Bin {b}: {count} images")


# step 3 - stratified 70/15/15 split
# stratify=forgery_bins keeps the bin proportions consistent across all splits
# random_state=42 locks the split so it's reproducible every run

# first cut: 70% train, 30% temp
train_idx, temp_idx = train_test_split(
    range(len(forged_image_paths)),
    test_size=0.30,
    stratify=forgery_bins,
    random_state=42
)

# second cut: temp split evenly into val and test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=forgery_bins[temp_idx],
    random_state=42
)

print(f"\nFinal split:")
print(f"  Train : {len(train_idx)} images (~70%)")
print(f"  Val   : {len(val_idx)} images (~15%)")
print(f"  Test  : {len(test_idx)} images (~15%)")
print(f"  Total : {len(train_idx) + len(val_idx) + len(test_idx)} images")


Dataset loaded:
  Authentic images   : 2377
  Forged images      : 2751
  Ground truth masks : 2751

Forgery density stats:
  Average : 4.89%
  Min     : 0.0006%
  Max     : 52.27%
  Bin 1: 684 images
  Bin 2: 394 images
  Bin 3: 715 images
  Bin 4: 958 images
  Bin 5: 0 images

Final split:
  Train : 1925 images (~70%)
  Val   : 413 images (~15%)
  Test  : 413 images (~15%)
  Total : 2751 images


In [5]:
# Block 2 - Augmentation pipeline and dataset class
#
# We were pretty conservative with augmentations here because two of our
# three agents work on subtle pixel-level signals. Colour jitter would
# corrupt the noise agent (it reads camera noise fingerprints), and adding
# fake JPEG compression would teach the frequency agent to detect the wrong
# thing. Cutout was also ruled out — it could erase the forged region
# entirely and give the model a wrong training label.
#
# What we kept: flips, 90-degree rotations, and small shifts/scales.
# These are all spatially safe — the mask just follows the image.
# Albumentations handles that automatically when you pass mask= to the transform.

train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),

    # border_mode=0 fills any gaps created by shifting with black pixels
    # using reflection would risk copying a forged region into the border
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=15,
        border_mode=0,
        p=0.70
    ),

    # encoder is pretrained on ImageNet so it needs these exact stats
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    ),
    ToTensorV2()
])

# no augmentation on val — want a stable consistent IoU each epoch
val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    ),
    ToTensorV2()
])


class ForgeryDataset(Dataset):
    """
    Loads forged image + binary mask pairs.
    Returns image [3, 256, 256] and mask [1, 256, 256] as float tensors.
    """

    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        # some microscopy images are saved as grayscale or RGBA
        # convert("RGB") forces everything to 3 channels consistently
        img = np.array(Image.open(self.image_paths[idx]).convert("RGB"))

        # masks can arrive in different shapes depending on how they were saved
        # squeeze + the ndim check handles [H,W], [1,H,W] and [H,W,1]
        mask = np.load(self.mask_paths[idx])
        mask = np.squeeze(mask)
        if mask.ndim == 3:
            mask = mask[0]

        # some masks store forged pixels as 255 rather than 1 — binarise
        mask = (mask > 0).astype(np.uint8)

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img  = augmented["image"]
            mask = augmented["mask"].float().unsqueeze(0)  # [1, 256, 256]

        return img, mask


# build path lists for each split using the indices from Block 1
train_imgs  = [forged_image_paths[i] for i in train_idx]
train_masks = [mask_paths[i]         for i in train_idx]

val_imgs    = [forged_image_paths[i] for i in val_idx]
val_masks   = [mask_paths[i]         for i in val_idx]

test_imgs   = [forged_image_paths[i] for i in test_idx]
test_masks  = [mask_paths[i]         for i in test_idx]

# physical batch is 4, gradient accumulation makes it behave like 16
# so only 4 images of activations sit on the GPU at any one time
BATCH_SIZE  = 4
ACCUM_STEPS = 4

train_loader = DataLoader(
    ForgeryDataset(train_imgs, train_masks, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True
)

val_loader = DataLoader(
    ForgeryDataset(val_imgs, val_masks, val_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Batch size    : {BATCH_SIZE}  (effective {BATCH_SIZE * ACCUM_STEPS} with accumulation)")


Train batches : 482
Val batches   : 104
Batch size    : 4  (effective 16 with accumulation)


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [6]:
# Block 3 - Domain pre-processors for the noise and frequency agents
#
# The problem with naive multi-agent systems is that if all agents see the
# same raw image they just learn the same features — you get three copies
# of one model. Our fix was to give each agent a genuinely different view
# of the image before any neural network touches it.
#
# Agent 1 gets raw RGB
# Agent 2 gets SRM noise residuals  — reveals camera noise fingerprint differences
# Agent 3 gets FFT frequency bands  — reveals boundary artefacts in frequency space
#
# Both extractors below have zero learnable parameters. We kept them fixed
# on purpose — if they were learnable they might drift into replicating
# what the RGB agent already does, defeating the whole point.


class SRMNoiseExtractor(nn.Module):
    """
    Extracts pixel-level noise residuals using Spatial Rich Model filters.
    Reference: Fridrich & Kodovsky, IEEE TIFS 2012.

    When a region is copy-pasted from another source it carries the noise
    fingerprint of that source — different camera, scanner or compression.
    SRM high-pass filters suppress all actual image content and leave only
    the noise behind. A forged region then shows a different noise texture
    from the surrounding authentic area.

    Three fixed 5x5 filters applied per colour channel, results averaged.
    Output: [B, 3, H, W] in [-1, 1]. Learnable params: 0.
    """

    def __init__(self):
        super().__init__()

        # second-order Laplacian — picks up fine-scale intensity variation
        k1 = torch.tensor([
            [ 0,  0,  0,  0,  0],
            [ 0, -1,  2, -1,  0],
            [ 0,  2, -4,  2,  0],
            [ 0, -1,  2, -1,  0],
            [ 0,  0,  0,  0,  0]
        ], dtype=torch.float32) / 4.0

        # stronger edge-aware filter — more sensitive to spatially correlated noise
        k2 = torch.tensor([
            [-1,  2, -2,  2, -1],
            [ 2, -6,  8, -6,  2],
            [-2,  8,-12,  8, -2],
            [ 2, -6,  8, -6,  2],
            [-1,  2, -2,  2, -1]
        ], dtype=torch.float32) / 12.0

        # simple horizontal gradient — copy-paste boundaries often show up here
        k3 = torch.tensor([
            [ 0,  0,  0,  0,  0],
            [ 0,  0,  0,  0,  0],
            [ 0,  1, -2,  1,  0],
            [ 0,  0,  0,  0,  0],
            [ 0,  0,  0,  0,  0]
        ], dtype=torch.float32) / 2.0

        # register_buffer: saved in state_dict but never touched by gradients
        self.register_buffer(
            'weight',
            torch.stack([k1, k2, k3]).unsqueeze(1)  # [3, 1, 5, 5]
        )

    def forward(self, x):
        # apply all three filters to each RGB channel separately then average
        # noise characteristics can differ per channel on some cameras
        outs = []
        for c in range(3):
            channel_response = F.conv2d(
                x[:, c:c+1],
                self.weight,
                padding=2  # keeps spatial size the same
            )
            outs.append(channel_response)

        out = torch.stack(outs).mean(0)

        # clamp outliers before they mess up BatchNorm stats downstream
        return torch.clamp(out, -2, 2) / 2.0


class FFTMagnitudeExtractor(nn.Module):
    """
    Converts image to frequency domain and extracts three magnitude bands.

    Copy-paste forgeries create hard spatial boundaries which show up as
    bursts of high-frequency energy in the Fourier domain. Authentic
    scientific images have consistent frequency signatures — a pasted region
    breaks that consistency in a measurable way.

    We split into three radial bands so the encoder can learn which frequency
    range is most informative for each forgery type rather than treating the
    whole spectrum as one signal.

    Output: [B, 3, H, W] in [0, 1]. Learnable params: 0.
    """

    def __init__(self, img_size=256):
        super().__init__()

        # radial distance from centre of the frequency plane
        # after fftshift, DC sits at centre — low freq near centre, high freq at edges
        cx = cy = img_size // 2
        y    = torch.arange(img_size).float() - cy
        x    = torch.arange(img_size).float() - cx
        dist = (y.unsqueeze(1)**2 + x.unsqueeze(0)**2).sqrt()

        r_low = img_size // 8  # 32px for 256 input
        r_mid = img_size // 4  # 64px for 256 input

        # precompute once and store — no need to rebuild these every forward pass
        self.register_buffer('low_mask',  (dist <= r_low).float()[None, None])
        self.register_buffer('mid_mask',  ((dist > r_low) & (dist <= r_mid)).float()[None, None])
        self.register_buffer('high_mask', (dist > r_mid).float()[None, None])

    def forward(self, x):
        # frequency artefacts appear in luminance not colour
        gray = x.mean(dim=1, keepdim=True)  # [B, 1, H, W]

        # 2D FFT then shift DC to centre (standard convention)
        fft = torch.fft.fftshift(torch.fft.fft2(gray, norm='ortho'))

        # log magnitude — without log the DC component drowns everything else out
        log_mag = torch.log(torch.abs(fft) + 1.0)

        # apply each band mask and stack into 3 channels
        out = torch.cat([
            log_mag * self.low_mask,
            log_mag * self.mid_mask,
            log_mag * self.high_mask
        ], dim=1)  # [B, 3, H, W]

        # normalise per image so the domain adapter always gets [0, 1]
        mn = out.amin(dim=[2, 3], keepdim=True)
        mx = out.amax(dim=[2, 3], keepdim=True)
        return (out - mn) / (mx - mn + 1e-8)


# quick sanity check
srm_extractor = SRMNoiseExtractor().to(device)
fft_extractor = FFTMagnitudeExtractor(img_size=256).to(device)

dummy   = torch.randn(2, 3, 256, 256).to(device)
srm_out = srm_extractor(dummy)
fft_out = fft_extractor(dummy)

print(f"SRM output : {tuple(srm_out.shape)}  range [{srm_out.min():.3f}, {srm_out.max():.3f}]")
print(f"FFT output : {tuple(fft_out.shape)}  range [{fft_out.min():.3f}, {fft_out.max():.3f}]")
print(f"SRM learnable params : {sum(p.numel() for p in srm_extractor.parameters())}")
print(f"FFT learnable params : {sum(p.numel() for p in fft_extractor.parameters())}")
# both should print 0


SRM output : (2, 3, 256, 256)  range [-1.000, 1.000]
FFT output : (2, 3, 256, 256)  range [0.000, 1.000]
SRM learnable params : 0
FFT learnable params : 0


In [7]:
# Block 4 - Domain adapters and shared encoder
#
# Two decisions worth explaining here.
#
# First, why one shared encoder instead of three separate backbones?
# Memory is the obvious answer — three ResNet-18s would be ~35M parameters
# plus three sets of Adam optimizer states, which the T4 cannot hold alongside
# three simultaneous forward passes. But there is also a training benefit:
# gradients from all three agents flow back through the same weights at once,
# forcing the encoder to learn features that work across RGB, noise and
# frequency simultaneously. It cannot overfit to just one domain.
#
# Second, why domain adapters at all?
# After Block 3 the three inputs have very different value ranges — RGB is
# roughly [-2, 2], SRM noise is sparse around [-1, 1], FFT bands are in [0, 1].
# ResNet's BatchNorm computes running mean and variance, so feeding three
# completely different distributions through the same encoder confuses its
# statistics. Each adapter is a tiny ~5K network that normalises its domain's
# signal to [0, 1] before the encoder sees it.


class DomainAdapter(nn.Module):
    """
    Small per-agent normaliser. Maps 3 input channels to 3 output channels in [0, 1].
    Conv(3->16, 3x3) -> BatchNorm -> GELU -> Conv(16->3, 1x1) -> Sigmoid
    Around 5K parameters per adapter — negligible next to the shared encoder.

    GELU rather than ReLU because SRM and FFT signals have many near-zero
    values that carry real meaning (no anomaly here). ReLU would hard-zero
    roughly half of those, GELU preserves them.
    """

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # 3x3 conv so each output looks at a small spatial neighbourhood
            # bias=False because BatchNorm below has its own learnable shift
            nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.GELU(),
            # 1x1 conv — channel mixing only, no spatial mixing at this stage
            nn.Conv2d(16, 3, kernel_size=1, bias=False),
            # sigmoid forces output to [0, 1] so the encoder always gets
            # a consistent value range regardless of which domain it came from
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


class SharedEncoder(nn.Module):
    """
    ResNet-18 pretrained on ImageNet, truncated after layer3.
    Returns main features [B, 256, 16, 16] and skip features [B, 64, 64, 64].

    We stop at layer3 because:
    - We need spatial maps, not a flat vector, so avgpool and fc are dropped
    - layer4 would give 8x8 which is too coarse for small forgery regions
    - 16x16 gives 256 spatial tokens per agent for the attention debate

    The 64x64 skip from layer1 is returned separately so the decoder can
    recover fine spatial detail that gets lost during downsampling.
    """

    def __init__(self):
        super().__init__()
        rn = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

        # stem: 7x7 conv stride=2 + maxpool stride=2 -> 256 to 64 spatially
        self.stem   = nn.Sequential(rn.conv1, rn.bn1, rn.relu, rn.maxpool)

        # layer1: no downsampling, stays at 64x64 — this becomes the skip connection
        self.layer1 = rn.layer1

        # layer2: stride=2, goes to 32x32
        self.layer2 = rn.layer2

        # layer3: stride=2, goes to 16x16 with 256 channels — main output
        self.layer3 = rn.layer3

    def forward(self, x):
        x    = self.stem(x)
        skip = self.layer1(x)   # [B, 64, 64, 64] — saved for decoder
        x    = self.layer2(skip)
        x    = self.layer3(x)   # [B, 256, 16, 16]
        return x, skip


# sanity check — confirm shapes before wiring everything together
rgb_adapter   = DomainAdapter().to(device)
noise_adapter = DomainAdapter().to(device)
freq_adapter  = DomainAdapter().to(device)
encoder       = SharedEncoder().to(device)

dummy_rgb   = torch.randn(2, 3, 256, 256).to(device)
dummy_noise = srm_extractor(dummy_rgb)
dummy_freq  = fft_extractor(dummy_rgb)

rgb_adapted   = rgb_adapter(dummy_rgb)
noise_adapted = noise_adapter(dummy_noise)
freq_adapted  = freq_adapter(dummy_freq)

rgb_enc,   skip_r = encoder(rgb_adapted)
noise_enc, skip_n = encoder(noise_adapted)
freq_enc,  skip_f = encoder(freq_adapted)

print("Adapter outputs (all should be [2, 3, 256, 256]):")
print(f"  RGB   : {tuple(rgb_adapted.shape)}  [{rgb_adapted.min():.2f}, {rgb_adapted.max():.2f}]")
print(f"  Noise : {tuple(noise_adapted.shape)}  [{noise_adapted.min():.2f}, {noise_adapted.max():.2f}]")
print(f"  Freq  : {tuple(freq_adapted.shape)}  [{freq_adapted.min():.2f}, {freq_adapted.max():.2f}]")

print("\nEncoder outputs:")
print(f"  Main : {tuple(rgb_enc.shape)}")
print(f"  Skip : {tuple(skip_r.shape)}")

adapter_params = sum(p.numel() for p in rgb_adapter.parameters())
encoder_params = sum(p.numel() for p in encoder.parameters())

print(f"\nAdapter params (one)   : {adapter_params:,}")
print(f"Shared encoder params  : {encoder_params/1e6:.2f}M")
print(f"Three separate ResNets : {encoder_params*3/1e6:.2f}M  — what we avoided")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 177MB/s] 


Adapter outputs (all should be [2, 3, 256, 256]):
  RGB   : (2, 3, 256, 256)  [0.11, 0.95]
  Noise : (2, 3, 256, 256)  [0.11, 0.87]
  Freq  : (2, 3, 256, 256)  [0.06, 0.88]

Encoder outputs:
  Main : (2, 256, 16, 16)
  Skip : (2, 64, 64, 64)

Adapter params (one)   : 512
Shared encoder params  : 2.78M
Three separate ResNets : 8.35M  — what we avoided


In [8]:
# Block 5 - Per-agent heads with spatial confidence maps
#
# After the shared encoder all three agents have identical shaped features
# [B, 256, 16, 16] — generic, not domain-specific yet. Each AgentHead
# compresses these to 128 channels and produces a spatial confidence map
# alongside the refined features.
#
# The confidence map is what separates this from a standard multi-agent setup.
# Most implementations combine agents with fixed weights — 1/3 each, everywhere.
# That's wrong for our problem. At a specific forgery location the noise agent
# might be 90% confident while the frequency agent is only 10% confident.
# Fixed weights would dilute the strong signal with the weak one.
# The confidence map lets each agent say how certain it is at each of the
# 256 spatial positions, and the debate in Block 6 uses that to weight
# contributions adaptively rather than equally.


class AgentHead(nn.Module):
    """
    Per-agent feature refinement and spatial confidence estimation.
    Three instances in the full model — same architecture, separate weights.

    Input:  [B, 256, 16, 16] shared encoder features
    Output: feat [B, 128, 16, 16] and conf [B, 1, 16, 16]

    We output 128 channels rather than keeping 256 because the debate
    concatenates all three agents — at 256 that would be 768 channels
    going into attention, making it 4x more expensive for no real gain.
    """

    def __init__(self):
        super().__init__()

        # two conv blocks: first compresses 256->128, second refines within
        # that 128-channel space. Two stages gives more expressive power
        # than a single block for picking up subtle forgery signals.
        self.refine = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.GELU()
        )

        # separate confidence branch — kept separate so it can specialise
        # in estimating certainty without fighting the feature refinement task.
        # 1x1 conv means confidence at each position depends only on that
        # position's features, no spatial smoothing.
        # Sigmoid keeps output in [0, 1] for use as a multiplicative gate.
        self.conf_head = nn.Sequential(
            nn.Conv2d(128, 1, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        feat = self.refine(x)       # [B, 128, 16, 16]
        conf = self.conf_head(feat) # [B,   1, 16, 16]
        return feat, conf


# three instances — same architecture but completely separate weights
# each head only ever sees features from its own domain so they diverge during training
rgb_head   = AgentHead().to(device)
noise_head = AgentHead().to(device)
freq_head  = AgentHead().to(device)

rgb_feat,   rgb_conf   = rgb_head(rgb_enc)
noise_feat, noise_conf = noise_head(noise_enc)
freq_feat,  freq_conf  = freq_head(freq_enc)

print("Agent head outputs:")
print(f"  RGB   feat : {tuple(rgb_feat.shape)}  conf : {tuple(rgb_conf.shape)}  [{rgb_conf.min():.3f}, {rgb_conf.max():.3f}]")
print(f"  Noise feat : {tuple(noise_feat.shape)}  conf : {tuple(noise_conf.shape)}  [{noise_conf.min():.3f}, {noise_conf.max():.3f}]")
print(f"  Freq  feat : {tuple(freq_feat.shape)}  conf : {tuple(freq_conf.shape)}  [{freq_conf.min():.3f}, {freq_conf.max():.3f}]")
print(f"\nParams per head : {sum(p.numel() for p in rgb_head.parameters()):,}")


Agent head outputs:
  RGB   feat : (2, 128, 16, 16)  conf : (2, 1, 16, 16)  [0.273, 0.862]
  Noise feat : (2, 128, 16, 16)  conf : (2, 1, 16, 16)  [0.217, 0.794]
  Freq  feat : (2, 128, 16, 16)  conf : (2, 1, 16, 16)  [0.127, 0.700]

Params per head : 443,009


In [9]:
# Block 6 - Cross-agent attention debate
#
# This is where the three agents stop working in isolation and actually
# communicate with each other.
#
# A simple weighted average only combines agents at the same spatial position.
# Position (3,7) in the RGB agent only ever talks to position (3,7) in the
# noise and frequency agents. But a noise anomaly at (2,8) and a frequency
# anomaly at (4,6) might both come from the same diagonal forgery boundary —
# a positional average completely misses that connection.
#
# Our fix: flatten each agent's 16x16 map into 256 tokens, concatenate all
# three agents into one sequence of 768 tokens, then run multi-head self-
# attention across all 768 at once. Every position in every agent can now
# attend to every other position in every other agent.
#
# The agent-type embeddings are a learnable 128-dim vector per agent added
# to all its tokens before attention runs. Without them the attention has no
# way to tell which agent a token came from — RGB position (3,7) and noise
# position (3,7) would look identical to it.


class CrossAgentAttention(nn.Module):
    """
    Three-way spatial debate via multi-head self-attention.

    Input:  fa, fb, fc each [B, 128, 16, 16]
    Output: oa, ob, oc each [B, 128, 16, 16] — same shape, enriched with
            evidence from all agents at all positions.

    Standard Pre-LN transformer block: norm before attention rather than
    after. More stable in early training because it keeps gradient magnitudes
    consistent across the 768-token joint sequence.
    """

    def __init__(self, dim=128, num_heads=4, dropout=0.10):
        super().__init__()

        # one 128-dim vector per agent, added to all tokens from that agent
        # initialised small so they don't swamp the content features early on
        self.agent_embeddings = nn.Parameter(torch.randn(3, dim) * 0.02)

        # 4 heads each operating on a 32-dim subspace (128/4)
        # different heads learn to attend to different types of cross-agent relationships
        self.attention = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(dim)

        # standard transformer FFN — expand then contract
        # attention gathers cross-agent evidence, FFN digests it
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim)
        )

        self.norm2 = nn.LayerNorm(dim)

    def forward(self, fa, fb, fc):
        B, C, H, W = fa.shape
        N = H * W  # 256 tokens per agent

        def flatten_and_tag(feat, idx):
            # [B, C, H, W] -> [B, N, C] then stamp with agent identity
            tokens = feat.flatten(2).transpose(1, 2)
            tokens = tokens + self.agent_embeddings[idx]
            return tokens

        # tag each agent's tokens with its identity embedding
        agent_a = flatten_and_tag(fa, 0)  # [B, 256, 128]
        agent_b = flatten_and_tag(fb, 1)
        agent_c = flatten_and_tag(fc, 2)

        # one joint sequence — every token can attend to all 767 others
        joint = torch.cat([agent_a, agent_b, agent_c], dim=1)  # [B, 768, 128]

        # pre-LN self-attention with residual
        normed   = self.norm1(joint)
        attn_out, _ = self.attention(query=normed, key=normed, value=normed)
        joint = joint + attn_out

        # FFN with residual
        joint = joint + self.ffn(self.norm2(joint))

        # split back into three agents and reshape to spatial maps
        oa, ob, oc = joint.split(N, dim=1)

        def unflatten(tokens):
            return tokens.transpose(1, 2).view(B, C, H, W)

        return unflatten(oa), unflatten(ob), unflatten(oc)


# sanity check
debate = CrossAgentAttention(dim=128, num_heads=4, dropout=0.10).to(device)

rgb_debated, noise_debated, freq_debated = debate(rgb_feat, noise_feat, freq_feat)

print("Debate output shapes:")
print(f"  RGB   : {tuple(rgb_debated.shape)}")
print(f"  Noise : {tuple(noise_debated.shape)}")
print(f"  Freq  : {tuple(freq_debated.shape)}")

# confirm features actually changed after debate
rgb_change   = (rgb_debated   - rgb_feat  ).abs().mean().item()
noise_change = (noise_debated - noise_feat).abs().mean().item()
freq_change  = (freq_debated  - freq_feat ).abs().mean().item()

print(f"\nMean change after debate (should all be > 0):")
print(f"  RGB {rgb_change:.6f}  Noise {noise_change:.6f}  Freq {freq_change:.6f}")
print(f"Debate params : {sum(p.numel() for p in debate.parameters()):,}")


Debate output shapes:
  RGB   : (2, 128, 16, 16)
  Noise : (2, 128, 16, 16)
  Freq  : (2, 128, 16, 16)

Mean change after debate (should all be > 0):
  RGB 0.168072  Noise 0.168838  Freq 0.168242
Debate params : 132,864


In [10]:
# Block 7 - Spatial prototype memory (CPU-resident)
#
# This module gives the model a memory bank that fills up during training
# with real forgery pattern embeddings. At inference it retrieves the most
# similar stored patterns and uses them to condition the current prediction.
#
# The bank lives on CPU permanently. It is updated with no_grad so there
# is no backpropagation through writes, meaning there is no reason to keep
# it on GPU — it would just waste VRAM we badly need for the three encoder
# passes and the attention computation.
#
# The tricky part: PyTorch's .to(device) moves everything including
# register_buffer contents. We override _apply() to force the bank back
# to CPU immediately after any device move. That way model.to(cuda) works
# correctly for all the learnable layers while the bank stays on CPU.


class SpatialPrototypeMemory(nn.Module):
    """
    CPU-resident forgery pattern bank with cross-attention retrieval.

    During training: writes forged-region embeddings to the bank.
    During inference: retrieves top-k similar patterns and conditions
    current features via cross-attention.
    """

    def __init__(self, embed_dim=128, bank_size=256, top_k=8):
        super().__init__()
        self.embed_dim = embed_dim
        self.bank_size = bank_size
        self.top_k     = top_k

        # these three stay on CPU permanently — _apply() below enforces this
        # register_buffer still includes them in state_dict for saving/loading
        self.register_buffer('bank',   torch.zeros(bank_size, embed_dim))
        self.register_buffer('ptr',    torch.tensor(0, dtype=torch.long))
        self.register_buffer('filled', torch.tensor(False))

        # these DO move to GPU — they are learnable layers
        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.mem_attn   = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=4,
            dropout=0.10,
            batch_first=True
        )
        self.norm = nn.LayerNorm(embed_dim)

    def _apply(self, fn):
        # called by .to(), .cuda(), .cpu(), .half() etc.
        # let the parent move everything, then force bank back to CPU
        super()._apply(fn)
        self.bank   = self.bank.cpu()
        self.ptr    = self.ptr.cpu()
        self.filled = self.filled.cpu()
        return self

    @torch.no_grad()
    def update(self, features, masks):
        """Write forged-region embeddings to the bank. No gradients here."""
        fh, fw = features.shape[2], features.shape[3]

        # downsample mask to match the 16x16 feature map
        m_ds = F.interpolate(
            masks.float(), size=(fh, fw), mode='bilinear', align_corners=False
        )

        for i in range(features.shape[0]):
            m = m_ds[i, 0]

            # skip authentic-only images
            if m.sum() < 1.0:
                continue

            f = features[i]

            # masked average pool — average only over forged pixel positions
            emb = (f * m.unsqueeze(0)).sum(dim=[1, 2]) / (m.sum() + 1e-6)

            # normalise then move to CPU for storage
            emb = F.normalize(emb, dim=0).cpu()

            # circular buffer write — both emb and bank are on CPU here
            idx            = self.ptr.item() % self.bank_size
            self.bank[idx] = emb
            self.ptr      += 1

            if not self.filled.item():
                self.filled = torch.tensor(True)

    def forward(self, features):
        """Retrieve similar patterns and condition current features."""
        # bank is empty early in training — just pass through
        if not self.filled.item():
            return features

        B, C, H, W = features.shape
        N = H * W

        # project features to a query vector, move to CPU for similarity search
        q      = self.query_proj(features.mean(dim=[2, 3]))  # [B, 128] GPU
        q_norm = F.normalize(q, dim=-1).detach().cpu()       # [B, 128] CPU

        # cosine similarity — both tensors on CPU, no device mismatch
        valid  = min(self.ptr.item(), self.bank_size)
        bank_n = F.normalize(self.bank[:valid], dim=-1)
        sim    = q_norm @ bank_n.t()                         # [B, valid] CPU

        k = min(self.top_k, valid)
        _, topk_idx = sim.topk(k, dim=-1)

        # fetch only the top-k embeddings and move them to GPU — tiny transfer
        retrieved = self.bank[topk_idx].to(features.device)  # [B, k, 128] GPU

        # every spatial position attends to the k retrieved forgery patterns
        f_flat    = features.flatten(2).transpose(1, 2)      # [B, 256, 128]
        ctx, _    = self.mem_attn(query=f_flat, key=retrieved, value=retrieved)
        f_flat    = self.norm(f_flat + ctx)

        return f_flat.transpose(1, 2).view(B, C, H, W)


# verify the device fix works
memory = SpatialPrototypeMemory(embed_dim=128, bank_size=256, top_k=8).to(device)

print(f"bank device      : {memory.bank.device}")   # must be cpu
print(f"query_proj device: {next(memory.query_proj.parameters()).device}")  # must be cuda

# test write
dummy_feat  = torch.randn(4, 128, 16, 16).to(device)
dummy_masks = torch.zeros(4, 1, 256, 256).to(device)
dummy_masks[0, 0, 40:90, 50:110] = 1.0
dummy_masks[2, 0, 20:60, 30:80]  = 1.0

memory.update(dummy_feat.detach(), dummy_masks)
print(f"Bank entries after write : {memory.ptr.item()}")
print(f"Bank still on CPU        : {memory.bank.device}")

# test forward
output = memory(dummy_feat)
print(f"Output shape : {tuple(output.shape)}  device : {output.device}")


bank device      : cpu
query_proj device: cuda:0
Bank entries after write : 2
Bank still on CPU        : cpu
Output shape : (4, 128, 16, 16)  device : cuda:0


In [11]:
# Block 8 - Decoder: 16x16 back to 256x256
#
# At this point we have three agent feature maps at 16x16 and a skip
# connection at 64x64 saved from encoder layer1. The decoder goes from
# that compressed representation back to a full pixel-level prediction.
#
# Two-stage upsampling: 16->64 then 64->256, each 4x via transposed conv.
# We use transposed convolutions rather than bilinear upsampling because
# they are learned — the kernel determines how each position spreads across
# the output, so the model can develop patterns specifically suited to
# recovering sharp forgery boundaries rather than generic smooth blobs.
#
# The skip connection matters because by the time features reach the decoder
# they have been compressed to 16x16 and fine spatial detail is gone. The
# 64x64 skip from encoder layer1 was only 4x downsampled so it still has
# that detail. Injecting it at the 64x64 stage gives the decoder precise
# boundary information exactly when it needs it.
#
# No sigmoid at the output — BCEWithLogitsLoss fuses sigmoid and BCE in a
# numerically stable way. We only apply sigmoid manually at inference.


class ForgeryDecoder(nn.Module):
    """
    16x16 -> 64x64 -> 256x256 decoder with skip connection.

    Input:  agent_feats list of 3x [B, 128, 16, 16] + skip [B, 64, 64, 64]
    Output: raw logit map [B, 1, 256, 256] — no sigmoid applied
    """

    def __init__(self):
        super().__init__()

        # concatenate three agents (3x128=384 channels) then compress to 256
        # 1x1 conv for channel mixing only — no spatial mixing at this stage
        self.agent_fuse = nn.Sequential(
            nn.Conv2d(128 * 3, 256, kernel_size=1, bias=False),
            nn.BatchNorm2d(256),
            nn.GELU()
        )

        # stage 1: 16->64 (4x), 256->128 channels
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=4),
            nn.BatchNorm2d(128),
            nn.GELU()
        )

        # fuse decoder (128ch) with encoder skip (64ch) at 64x64
        # 3x3 here so each position can look at its neighbours when merging
        self.skip_fuse = nn.Sequential(
            nn.Conv2d(128 + 64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.GELU()
        )

        # stage 2: 64->256 (4x), 128->64 channels
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=4),
            nn.BatchNorm2d(64),
            nn.GELU()
        )

        # final prediction — no sigmoid, raw logit output
        self.pred_head = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1, bias=False),
            nn.GELU(),
            nn.Conv2d(32, 1, kernel_size=1)
        )

    def forward(self, agent_feats, skip):
        x = torch.cat(agent_feats, dim=1)  # [B, 384, 16, 16]
        x = self.agent_fuse(x)             # [B, 256, 16, 16]

        x = self.up1(x)                    # [B, 128, 64, 64]

        # inject skip — encoder's fine-grained spatial detail
        x = torch.cat([x, skip], dim=1)   # [B, 192, 64, 64]
        x = self.skip_fuse(x)             # [B, 128, 64, 64]

        x = self.up2(x)                   # [B, 64, 256, 256]
        return self.pred_head(x)          # [B, 1, 256, 256]


# sanity check
decoder = ForgeryDecoder().to(device)

fake_feats = [torch.randn(2, 128, 16, 16).to(device) for _ in range(3)]
fake_skip  = torch.randn(2, 64, 64, 64).to(device)

logit_map = decoder(fake_feats, fake_skip)

print(f"Decoder output : {tuple(logit_map.shape)}")
print(f"Logit range    : [{logit_map.min():.3f}, {logit_map.max():.3f}]")
print(f"After sigmoid  : [{logit_map.sigmoid().min():.4f}, {logit_map.sigmoid().max():.4f}]")
print(f"Decoder params : {sum(p.numel() for p in decoder.parameters()):,}")


Decoder output : (2, 1, 256, 256)
Logit range    : [-0.831, 0.426]
After sigmoid  : [0.3033, 0.6049]
Decoder params : 994,657


In [12]:
# Block 9 - Full AgenticForensicNet assembly
#
# This block wires together everything from Blocks 3-8 into one nn.Module.
# No new architecture here — just the complete pipeline connected in order.
#
# Forward pass overview:
#   1. SRM and FFT generate noise and frequency views of the input
#   2. Domain adapters normalise all three inputs to [0, 1]
#   3. Shared encoder runs three times, one pass per domain
#   4. Agent heads refine features and produce spatial confidence maps
#   5. Cross-agent attention lets all three agents communicate
#   6. Confidence-gated residual: feat = feat + conf * debated_feat
#   7. Prototype memory writes during training, retrieves always
#   8. Decoder goes from 16x16 back to 256x256


class AgenticForensicNet(nn.Module):
    """Three-agent forensic segmentation network with prototype memory."""

    def __init__(self):
        super().__init__()

        # fixed signal processors — zero learnable params
        self.srm = SRMNoiseExtractor()
        self.fft = FFTMagnitudeExtractor(img_size=256)

        # one tiny adapter per agent to normalise each domain to [0, 1]
        self.rgb_adapter   = DomainAdapter()
        self.noise_adapter = DomainAdapter()
        self.freq_adapter  = DomainAdapter()

        # one shared ResNet-18 for all three — saves ~23M params vs separate encoders
        self.encoder = SharedEncoder()

        # three heads, same architecture but completely separate weights
        self.rgb_head   = AgentHead()
        self.noise_head = AgentHead()
        self.freq_head  = AgentHead()

        # cross-agent attention — 768 joint tokens (3 agents x 256 positions)
        self.debate = CrossAgentAttention(dim=128, num_heads=4, dropout=0.10)

        # CPU-resident prototype bank
        self.memory = SpatialPrototypeMemory(embed_dim=128, bank_size=256, top_k=8)

        # two-stage decoder with skip connection
        self.decoder = ForgeryDecoder()

    def forward(self, x, masks=None):
        """
        x     : [B, 3, 256, 256] input images
        masks : [B, 1, H, W] ground truth — only needed during training for
                memory bank writes. Pass None at validation/inference.

        Returns logit map [B, 1, 256, 256] and three confidence maps [B, 1, 16, 16].
        """

        # step 1 — three different forensic views of the same image
        rgb_in   = self.rgb_adapter(x)
        noise_in = self.noise_adapter(self.srm(x))
        freq_in  = self.freq_adapter(self.fft(x))

        # step 2 — shared encoder, three passes
        rgb_enc,   skip_r = self.encoder(rgb_in)
        noise_enc, skip_n = self.encoder(noise_in)
        freq_enc,  skip_f = self.encoder(freq_in)

        # average the three skip connections — blends fine-grained detail
        # from all three domain perspectives for the decoder
        skip = (skip_r + skip_n + skip_f) / 3.0

        # step 3 — per-agent heads: compress to 128ch and get confidence maps
        rgb_feat,   rgb_conf   = self.rgb_head(rgb_enc)
        noise_feat, noise_conf = self.noise_head(noise_enc)
        freq_feat,  freq_conf  = self.freq_head(freq_enc)

        # step 4 — cross-agent debate: all 768 tokens communicate
        rgb_d, noise_d, freq_d = self.debate(rgb_feat, noise_feat, freq_feat)

        # step 5 — confidence-gated residual
        # high confidence positions take more from the debate,
        # low confidence positions stay closer to their original features
        rgb_feat   = rgb_feat   + rgb_conf   * rgb_d
        noise_feat = noise_feat + noise_conf * noise_d
        freq_feat  = freq_feat  + freq_conf  * freq_d

        # step 6 — prototype memory
        consensus = (rgb_feat + noise_feat + freq_feat) / 3.0

        # only write to bank during training — never at validation (no leakage)
        if self.training and masks is not None:
            self.memory.update(consensus.detach(), masks)

        consensus = self.memory(consensus)

        # add memory-conditioned consensus back to each agent
        rgb_feat   = rgb_feat   + consensus
        noise_feat = noise_feat + consensus
        freq_feat  = freq_feat  + consensus

        # step 7 — decode to full resolution
        final_mask = self.decoder([rgb_feat, noise_feat, freq_feat], skip)

        return final_mask, rgb_conf, noise_conf, freq_conf


# build and verify
torch.cuda.empty_cache()
model = AgenticForensicNet().to(device)

def count_params(m):
    return sum(p.numel() for p in m.parameters())

total     = count_params(model)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("AgenticForensicNet parameter breakdown:")
print(f"  SRM / FFT (fixed)  : {count_params(model.srm):>10,}")
print(f"  Domain adapters x3 : {count_params(model.rgb_adapter)*3:>10,}")
print(f"  Shared encoder     : {count_params(model.encoder):>10,}")
print(f"  Agent heads x3     : {count_params(model.rgb_head)*3:>10,}")
print(f"  Debate module      : {count_params(model.debate):>10,}")
print(f"  Memory module      : {count_params(model.memory):>10,}")
print(f"  Decoder            : {count_params(model.decoder):>10,}")
print(f"  TOTAL              : {total:>10,}  ({total/1e6:.2f}M)")

# full forward pass test
dummy_images = torch.randn(4, 3, 256, 256).to(device)
dummy_masks  = torch.zeros(4, 1, 256, 256).to(device)
dummy_masks[0, 0, 60:100, 70:130] = 1.0
dummy_masks[2, 0, 20:50,  30:80]  = 1.0

model.train()
with torch.no_grad():
    final_mask, rgb_conf, noise_conf, freq_conf = model(dummy_images, dummy_masks)

print(f"\nForward pass output shapes:")
print(f"  final_mask : {tuple(final_mask.shape)}")
print(f"  rgb_conf   : {tuple(rgb_conf.shape)}")
print(f"  GPU memory : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"  Bank entries after forward : {model.memory.ptr.item()}")
print(f"  Bank device                : {model.memory.bank.device}")

# confirm inference works without masks
model.eval()
with torch.no_grad():
    out, _, _, _ = model(dummy_images)
print(f"\nInference (no masks) output : {tuple(out.shape)}")


AgenticForensicNet parameter breakdown:
  SRM / FFT (fixed)  :          0
  Domain adapters x3 :      1,536
  Shared encoder     :  2,782,784
  Agent heads x3     :  1,329,027
  Debate module      :    132,864
  Memory module      :     82,816
  Decoder            :    994,657
  TOTAL              :  5,323,684  (5.32M)

Forward pass output shapes:
  final_mask : (4, 1, 256, 256)
  rgb_conf   : (4, 1, 16, 16)
  GPU memory : 0.54 GB
  Bank entries after forward : 2
  Bank device                : cpu

Inference (no masks) output : (4, 1, 256, 256)


In [13]:
# Block 10 - Combined Focal + Dice loss
#
# Standard BCE does not work here. With a 150:1 class imbalance a model
# that predicts every pixel as authentic gets BCE ≈ 0.05 which looks fine,
# so training happily converges to a model that ignores forgeries entirely.
#
# Focal Loss fixes the per-pixel problem. It multiplies each pixel's BCE
# by (1 - pt)^gamma where pt is the model's probability for the correct class.
# A confident correct prediction on an easy authentic pixel (pt=0.99) gets
# weight 0.0001 — almost no gradient. A hard forged pixel at pt=0.5 gets
# weight 0.25 — full gradient. Hard examples dominate training.
# alpha=0.75 additionally up-weights the forged class on top of that.
#
# Dice Loss fixes the metric mismatch. BCE optimises per-pixel accuracy
# but we evaluate with IoU which measures region overlap. These are not
# the same objective. Dice directly optimises region overlap and ignores
# true negatives entirely so the 95% authentic background contributes
# nothing to it. Together the two losses cover what BCE misses.


class CombinedLoss(nn.Module):
    """
    Focal Loss + Dice Loss for imbalanced binary segmentation.

    alpha       : class weight for forged pixels (0.75 = 3x weight vs authentic)
    gamma       : focal focusing strength — gamma=2 from RetinaNet (Lin et al. 2017)
    dice_weight : relative contribution of Dice vs Focal, 1.0 = equal
    """

    def __init__(self, alpha=0.75, gamma=2.0, dice_weight=1.0):
        super().__init__()
        self.alpha       = alpha
        self.gamma       = gamma
        self.dice_weight = dice_weight

    def forward(self, pred, target):
        # pred   : [B, 1, H, W] raw logits — no sigmoid yet
        # target : [B, 1, H, W] binary ground truth

        # focal loss
        # BCEWithLogitsLoss applies sigmoid internally in a numerically stable way
        bce   = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt    = torch.exp(-bce)  # probability for the correct class
        focal = self.alpha * (1.0 - pt) ** self.gamma * bce
        focal = focal.mean()

        # dice loss — needs sigmoid first, no stable logit form like BCE
        pred_s = pred.sigmoid()
        smooth = 1e-6
        inter  = (pred_s * target).sum()
        dice   = 1.0 - (2.0 * inter + smooth) / (pred_s.sum() + target.sum() + smooth)

        return focal + self.dice_weight * dice


criterion = CombinedLoss(alpha=0.75, gamma=2.0, dice_weight=1.0).to(device)
print(f"CombinedLoss: alpha={criterion.alpha}, gamma={criterion.gamma}, dice_weight={criterion.dice_weight}")


# sanity checks
# important: create test tensors directly on device in one step
# doing torch.randn(..., requires_grad=True).to(device) creates a non-leaf
# tensor and .grad will be None after backward — caught us out earlier

# test 1: near-perfect prediction should give near-zero loss
perfect_pred   = torch.tensor([[[[10.0, -10.0], [-10.0, -10.0]]]], device=device, requires_grad=True)
perfect_target = torch.tensor([[[[1.0,   0.0],  [0.0,   0.0]]]], device=device)
print(f"Test 1 — near-perfect : {criterion(perfect_pred, perfect_target).item():.6f}  (expect ~0)")

# test 2: completely wrong should give high loss
wrong_pred   = torch.tensor([[[[10.0, 10.0], [10.0, 10.0]]]], device=device, requires_grad=True)
wrong_target = torch.tensor([[[[0.0,  0.0],  [0.0,  0.0]]]], device=device)
print(f"Test 2 — all wrong    : {criterion(wrong_pred, wrong_target).item():.6f}  (expect high)")

# test 3: always predicts authentic — the failure mode BCE falls into
always_auth  = torch.full((2, 1, 256, 256), -5.0, device=device, requires_grad=True)
imb_target   = torch.zeros(2, 1, 256, 256, device=device)
imb_target[0, 0, 60:80, 70:100] = 1.0
print(f"Test 3 — ignores all  : {criterion(always_auth, imb_target).item():.6f}  (expect not ~0)")

# test 4: gradient flow — most important check
grad_pred   = torch.randn(4, 1, 256, 256, device=device, requires_grad=True)
grad_target = torch.zeros(4, 1, 256, 256, device=device)
grad_target[0, 0, 50:90, 60:110] = 1.0
grad_target[2, 0, 20:50, 30:80]  = 1.0

loss = criterion(grad_pred, grad_target)
loss.backward()

print(f"Test 4 — gradients    : {grad_pred.grad is not None}  "
      f"mean={grad_pred.grad.abs().mean():.8f}  loss={loss.item():.4f}")
print("All tests passed.")


CombinedLoss: alpha=0.75, gamma=2.0, dice_weight=1.0
Test 1 — near-perfect : 0.000091  (expect ~0)
Test 2 — all wrong    : 8.499352  (expect high)
Test 3 — ignores all  : 1.011523  (expect not ~0)
Test 4 — gradients    : True  mean=0.00000117  loss=1.2340
All tests passed.


In [14]:
# Block 11 - Optimiser, scheduler, AMP scaler
#
# AdamW at lr=1e-4 — standard for fine-tuning pretrained models. Going
# higher would corrupt the ImageNet encoder weights in the first few batches.
# weight_decay=1e-4 adds L2 regularisation which helps given our small dataset.
#
# Cosine annealing smoothly decays lr from 1e-4 to near-zero over all epochs.
# We prefer this over step decay which causes abrupt loss jumps at each milestone.
# T_max=NUM_EPOCHS means lr hits near-zero exactly at the last epoch.
#
# GradScaler enables AMP — fp16 forward pass halves activation memory which
# is critical for fitting three encoder passes on the T4. The scaler handles
# numerically stable backward by scaling the loss up before backward and
# dividing gradients back down before the optimizer step. It also detects
# overflow and automatically skips bad updates.

import math

torch.cuda.empty_cache()
print(f"GPU memory before setup : {torch.cuda.memory_allocated()/1e9:.2f} GB")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

print(f"AdamW: lr={optimizer.param_groups[0]['lr']}  "
      f"wd={optimizer.param_groups[0]['weight_decay']}  "
      f"params={sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

NUM_EPOCHS = 40

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-7  # small floor so lr never fully stalls
)

# show how lr decays across training
print("Cosine LR schedule:")
for ep in [0, 10, 20, 30, 40]:
    lr = 1e-7 + 0.5 * (1e-4 - 1e-7) * (1 + math.cos(math.pi * ep / NUM_EPOCHS))
    print(f"  epoch {ep:2d} : {lr:.2e}")

scaler = GradScaler(
    init_scale=2**16,
    growth_factor=2.0,
    backoff_factor=0.5,
    growth_interval=2000
)

ACCUM_STEPS = 4  # physical batch 4 x 4 steps = effective batch 16

print(f"\nTraining setup:")
print(f"  Epochs          : {NUM_EPOCHS}")
print(f"  Batch size      : {BATCH_SIZE}  (effective {BATCH_SIZE * ACCUM_STEPS} with accumulation)")
print(f"  Precision       : fp16 forward / fp32 gradients")
print(f"  GPU after setup : {torch.cuda.memory_allocated()/1e9:.2f} GB")


GPU memory before setup : 0.55 GB
AdamW: lr=0.0001  wd=0.0001  params=5,323,684
Cosine LR schedule:
  epoch  0 : 1.00e-04
  epoch 10 : 8.54e-05
  epoch 20 : 5.01e-05
  epoch 30 : 1.47e-05
  epoch 40 : 1.00e-07

Training setup:
  Epochs          : 40
  Batch size      : 4  (effective 16 with accumulation)
  Precision       : fp16 forward / fp32 gradients
  GPU after setup : 0.55 GB


/tmp/ipykernel_55/3542449081.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(


In [15]:
# Block 12 - Training loop
#
# Three things keeping memory under control:
#
# AMP — forward pass in fp16 halves activation memory. GradScaler handles
# numerically stable backward in fp32.
#
# Gradient accumulation — physical batch is 4, we accumulate 4 steps before
# each optimizer step giving effective batch 16. Critical: divide loss by
# ACCUM_STEPS before backward, otherwise accumulated gradients are 4x too
# large and training explodes.
#
# empty_cache() every accumulation window — clears reserved but unused GPU
# memory. Doesn't free active tensors, just reduces fragmentation.
#
# Memory bank: masks are passed during training so the bank gets written.
# No masks at validation — bank stays read-only, no data leakage.

print("=" * 60)
print("Starting Training — AgenticForensicNet")
print("=" * 60)
print(f"  Epochs         : {NUM_EPOCHS}")
print(f"  Physical batch : {BATCH_SIZE}  (effective {BATCH_SIZE * ACCUM_STEPS})")
print(f"  Train batches  : {len(train_loader)} per epoch")
print(f"  Val batches    : {len(val_loader)} per epoch")
print(f"  GPU memory     : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("=" * 60)

best_val_iou       = 0.0
best_epoch         = 0
train_loss_history = []
val_iou_history    = []

for epoch in range(NUM_EPOCHS):

    # training phase
    model.train()
    running_loss = 0.0
    num_batches  = 0
    epoch_start  = time.time()
    optimizer.zero_grad()

    for step, (images, masks) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device,  non_blocking=True).float()

        with autocast():
            # pass masks so memory bank gets written during training
            final_mask, _, _, _ = model(images, masks)

            # divide by ACCUM_STEPS before backward — without this
            # gradients accumulate 4x too large and training diverges
            loss = criterion(final_mask, masks) / ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUM_STEPS
        num_batches  += 1

        if (step + 1) % ACCUM_STEPS == 0:
            # unscale before clipping so we clip true gradient magnitudes
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()

    scheduler.step()

    # validation phase — no masks passed, bank stays read-only
    model.eval()
    val_iou_list = []

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device, non_blocking=True)
            masks  = masks.to(device,  non_blocking=True).float()

            with autocast():
                final_mask, _, _, _ = model(images)

            pred = (final_mask.sigmoid() > 0.5).float()

            intersection = (pred * masks).sum()
            union        = pred.sum() + masks.sum() - intersection
            iou          = (intersection + 1e-6) / (union + 1e-6)
            val_iou_list.append(iou.item())

    val_iou  = np.mean(val_iou_list)
    avg_loss = running_loss / num_batches

    train_loss_history.append(avg_loss)
    val_iou_history.append(val_iou)

    bank_filled = min(model.memory.ptr.item(), 256)
    current_lr  = scheduler.get_last_lr()[0]
    epoch_time  = time.time() - epoch_start

    print(
        f"Epoch [{epoch+1:2d}/{NUM_EPOCHS}]  "
        f"Loss: {avg_loss:.4f}  |  "
        f"Val IoU: {val_iou:.4f}  |  "
        f"GPU: {torch.cuda.memory_allocated()/1e9:.2f}GB  |  "
        f"Bank: {bank_filled:3d}/256  |  "
        f"LR: {current_lr:.2e}  |  "
        f"Time: {epoch_time/60:.1f}min"
    )

    # save whenever validation IoU improves — gives the best generalising
    # checkpoint rather than just whatever the final epoch produced
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        best_epoch   = epoch + 1

        torch.save({
            'epoch'               : epoch + 1,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict'   : scaler.state_dict(),
            'val_iou'             : val_iou,
            'train_loss'          : avg_loss,
            'memory_bank_entries' : bank_filled,
            'loss_config'         : {
                'alpha'      : criterion.alpha,
                'gamma'      : criterion.gamma,
                'dice_weight': criterion.dice_weight
            }
        }, '/kaggle/working/best_model.pt')

        print(f"  -> New best saved  (Val IoU: {val_iou:.4f})")


# post-training summary
print("\n" + "=" * 60)
print("Training Complete")
print("=" * 60)
print(f"  Best Val IoU  : {best_val_iou:.4f}  (epoch {best_epoch})")
print(f"  Final loss    : {train_loss_history[-1]:.4f}")
print(f"  Final Val IoU : {val_iou_history[-1]:.4f}")
print(f"  Bank entries  : {min(model.memory.ptr.item(), 256)}/256")

if len(train_loss_history) >= 5:
    early = np.mean(train_loss_history[:5])
    late  = np.mean(train_loss_history[-5:])
    trend = "decreased" if late < early else "did not decrease"
    print(f"  Loss trend    : {early:.4f} -> {late:.4f}  ({trend})")

if len(val_iou_history) >= 5:
    early = np.mean(val_iou_history[:5])
    late  = np.mean(val_iou_history[-5:])
    trend = "improved" if late > early else "no improvement"
    print(f"  IoU trend     : {early:.4f} -> {late:.4f}  ({trend})")


Starting Training — AgenticForensicNet
  Epochs         : 40
  Physical batch : 4  (effective 16)
  Train batches  : 482 per epoch
  Val batches    : 104 per epoch
  GPU memory     : 0.55 GB


/tmp/ipykernel_55/3439446606.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_55/3439446606.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [ 1/40]  Loss: 0.8920  |  Val IoU: 0.2343  |  GPU: 0.62GB  |  Bank: 256/256  |  LR: 9.98e-05  |  Time: 1.3min
  -> New best saved  (Val IoU: 0.2343)
Epoch [ 2/40]  Loss: 0.7695  |  Val IoU: 0.2582  |  GPU: 0.62GB  |  Bank: 256/256  |  LR: 9.94e-05  |  Time: 1.2min
  -> New best saved  (Val IoU: 0.2582)
Epoch [ 3/40]  Loss: 0.7198  |  Val IoU: 0.2603  |  GPU: 0.62GB  |  Bank: 256/256  |  LR: 9.86e-05  |  Time: 1.1min
  -> New best saved  (Val IoU: 0.2603)
Epoch [ 4/40]  Loss: 0.7097  |  Val IoU: 0.2622  |  GPU: 0.62GB  |  Bank: 256/256  |  LR: 9.76e-05  |  Time: 1.2min
  -> New best saved  (Val IoU: 0.2622)
Epoch [ 5/40]  Loss: 0.6975  |  Val IoU: 0.2782  |  GPU: 0.62GB  |  Bank: 256/256  |  LR: 9.62e-05  |  Time: 1.2min
  -> New best saved  (Val IoU: 0.2782)
Epoch [ 6/40]  Loss: 0.6941  |  Val IoU: 0.2686  |  GPU: 0.62GB  |  Bank: 256/256  |  LR: 9.46e-05  |  Time: 1.1min
Epoch [ 7/40]  Loss: 0.6719  |  Val IoU: 0.2750  |  GPU: 0.62GB  |  Bank: 256/256  |  LR: 9.26e-05  |  Time: 

In [16]:
# Block 13 - Final evaluation with Test-Time Augmentation (TTA)
#
# Rather than one forward pass per image, TTA runs four augmented passes
# (original, horizontal flip, vertical flip, both flips), undoes each flip
# before averaging the probability maps. The average is more stable than
# any single prediction — random spatial noise cancels out across views.
# This is especially useful for small forgery regions near image edges.
#
# weights_only=False on torch.load is needed for PyTorch 2.6+ which changed
# the default from False to True. Our checkpoint contains numpy scalars from
# the scheduler state which the new default blocks. Safe to disable here
# since this is our own checkpoint from the same session.

print("Loading best checkpoint...")

checkpoint = torch.load(
    '/kaggle/working/best_model.pt',
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint['model_state_dict'])

print(f"  Epoch    : {checkpoint['epoch']}")
print(f"  Val IoU  : {checkpoint['val_iou']:.4f}")
print(f"  Loss     : {checkpoint['train_loss']:.4f}")
print(f"  Bank     : {checkpoint.get('memory_bank_entries', 'N/A')}/256")


@torch.no_grad()
def predict_with_tta(model, image):
    """Four-pass TTA: original + h-flip + v-flip + both. Returns averaged probs."""
    model.eval()
    predictions   = []
    augmentations = [(False, False), (True, False), (False, True), (True, True)]

    for flip_h, flip_v in augmentations:
        aug = image.clone()
        if flip_h:
            aug = torch.flip(aug, dims=[3])
        if flip_v:
            aug = torch.flip(aug, dims=[2])

        with torch.autocast(device_type="cuda"):
            logit, _, _, _ = model(aug)
        prob = logit.sigmoid()

        # undo the flip so all predictions are in the original orientation
        if flip_h:
            prob = torch.flip(prob, dims=[3])
        if flip_v:
            prob = torch.flip(prob, dims=[2])

        predictions.append(prob)

    return torch.stack(predictions).mean(dim=0)


print("\nRunning final evaluation with TTA...")
model.eval()
val_iou_list  = []
val_dice_list = []

for batch_idx, (images, masks) in enumerate(val_loader):
    images = images.to(device, non_blocking=True)
    masks  = masks.to(device,  non_blocking=True).float()

    for i in range(images.shape[0]):
        single_image = images[i:i+1]
        single_mask  = masks[i:i+1]

        avg_prob = predict_with_tta(model, single_image)
        pred     = (avg_prob > 0.5).float()

        intersection = (pred * single_mask).sum()
        union        = pred.sum() + single_mask.sum() - intersection
        iou          = (intersection + 1e-6) / (union + 1e-6)
        val_iou_list.append(iou.item())

        dice = (2.0 * intersection + 1e-6) / (pred.sum() + single_mask.sum() + 1e-6)
        val_dice_list.append(dice.item())

    if (batch_idx + 1) % 20 == 0:
        print(f"  Processed {batch_idx+1}/{len(val_loader)} batches")


final_iou     = np.mean(val_iou_list)
final_dice    = np.mean(val_dice_list)
final_iou_std = np.std(val_iou_list)
val_arr       = np.array(val_iou_list)

print("\n" + "=" * 60)
print("FINAL RESULTS — AgenticForensicNet + TTA")
print("=" * 60)
print(f"  Validation IoU  : {final_iou:.4f} +/- {final_iou_std:.4f}")
print(f"  Validation Dice : {final_dice:.4f}")
print("=" * 60)

print(f"\nPer-image IoU distribution:")
print(f"  Mean   : {val_arr.mean():.4f}")
print(f"  Median : {np.median(val_arr):.4f}")
print(f"  Min    : {val_arr.min():.4f}")
print(f"  Max    : {val_arr.max():.4f}")
print(f"  IoU > 0.3 : {(val_arr > 0.3).sum()} / {len(val_arr)}")
print(f"  IoU = 0.0 : {(val_arr < 0.01).sum()} / {len(val_arr)}")
print("\nDone.")


Loading best checkpoint...
  Epoch    : 16
  Val IoU  : 0.3143
  Loss     : 0.6143
  Bank     : 256/256

Running final evaluation with TTA...
  Processed 20/104 batches
  Processed 40/104 batches
  Processed 60/104 batches
  Processed 80/104 batches
  Processed 100/104 batches

FINAL RESULTS — AgenticForensicNet + TTA
  Validation IoU  : 0.2252 +/- 0.2336
  Validation Dice : 0.3129

Per-image IoU distribution:
  Mean   : 0.2252
  Median : 0.1976
  Min    : 0.0000
  Max    : 1.0000
  IoU > 0.3 : 150 / 413
  IoU = 0.0 : 149 / 413

Done.
